
# 22 — Wind-Calm Pollution Accumulation Test

Replacement notebook with retry logic, caching, and safe handling when scene-level NO₂ stats are missing.


In [ ]:

SITES = [
    {"site_id": "EST", "site_name": "Eastbourne",   "lat": 50.7680,  "lon": 0.2900},
    {"site_id": "NHV", "site_name": "Newhaven ERF", "lat": 50.80208, "lon": 0.04961},
    {"site_id": "LWS", "site_name": "Lewes",        "lat": 50.8739,  "lon": 0.0110},
]

DATE_FROM = "2024-12-26"
DATE_TO   = "2024-12-31"
CALM_WIND_THRESHOLD_MS = 2.0

SCENE_STATS_DIR = "outputs/s5p_stack"
OUTPUT_DIR = "outputs/wind_calm_test"


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import json, time
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
CACHE_DIR = Path(OUTPUT_DIR) / "weather_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def fetch_weather_with_retry(lat, lon, start_date, end_date, hourly_vars, retries=4, timeout=45):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": hourly_vars,
        "timezone": "UTC",
    }
    last_err = None
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            print(f"Weather fetch failed (attempt {attempt+1}/{retries}): {e}")
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Open-Meteo request failed after {retries} attempts: {last_err}")

def weather_cache_path(cache_dir: Path, lat, lon, start_date, end_date, hourly_vars):
    safe = f"{lat}_{lon}_{start_date}_{end_date}_{hourly_vars}".replace(",", "_").replace("/", "_").replace(":", "_")
    return cache_dir / f"{safe}.json"

def fetch_weather_cached(cache_dir: Path, lat, lon, start_date, end_date, hourly_vars):
    p = weather_cache_path(cache_dir, lat, lon, start_date, end_date, hourly_vars)
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    j = fetch_weather_with_retry(lat, lon, start_date, end_date, hourly_vars)
    p.write_text(json.dumps(j), encoding="utf-8")
    return j

print("UTC now:", datetime.now(timezone.utc).isoformat())
print("OUTPUT_DIR:", Path(OUTPUT_DIR).resolve())


In [ ]:

def fetch_weather(lat, lon, start_date, end_date):
    j = fetch_weather_cached(
        CACHE_DIR,
        lat,
        lon,
        start_date,
        end_date,
        "wind_speed_10m,wind_direction_10m,cloud_cover",
    )
    h = j.get("hourly", {})
    return pd.DataFrame({
        "time_utc": pd.to_datetime(h.get("time", []), utc=True, errors="coerce"),
        "wind_speed_ms": h.get("wind_speed_10m", []),
        "wind_dir_deg": h.get("wind_direction_10m", []),
        "cloud_cover_pct": h.get("cloud_cover", []),
    })

wx_rows = []
for site in SITES:
    w = fetch_weather(site["lat"], site["lon"], DATE_FROM, DATE_TO)
    w["site_id"] = site["site_id"]
    w["site_name"] = site["site_name"]
    w["date"] = w["time_utc"].dt.date.astype("string")
    wx_rows.append(w)

wx = pd.concat(wx_rows, ignore_index=True)

daily_wx = (
    wx.groupby(["site_id", "site_name", "date"], dropna=False)
    .agg(
        mean_wind_speed_ms=("wind_speed_ms", "mean"),
        mean_cloud_cover_pct=("cloud_cover_pct", "mean"),
        mean_wind_dir_deg=("wind_dir_deg", "mean"),
    )
    .reset_index()
)

print("Weather rows:", len(wx))
display(daily_wx.head(20))


In [ ]:

stats_rows = []

for site in SITES:
    p_csv = Path(SCENE_STATS_DIR) / f"{site['site_id']}_scene_level_no2_window_stats.csv"
    p_pq = Path(SCENE_STATS_DIR) / f"{site['site_id']}_scene_level_no2_window_stats.parquet"
    p = p_csv if p_csv.exists() else p_pq if p_pq.exists() else None

    if p is None:
        continue

    df = pd.read_csv(p) if p.suffix == ".csv" else pd.read_parquet(p)

    if "start_utc" in df.columns:
        df["start_utc"] = pd.to_datetime(df["start_utc"], utc=True, errors="coerce")
        df["date"] = df["start_utc"].dt.date.astype("string")

    df["site_id"] = site["site_id"]
    df["site_name"] = site["site_name"]
    stats_rows.append(df)

stats = pd.concat(stats_rows, ignore_index=True) if stats_rows else pd.DataFrame()

print("Stats rows:", len(stats))
if not stats.empty:
    display(stats.head(20))
else:
    print("No scene-level NO2 stats found — notebook will run in screening mode.")


In [ ]:

calm_days = daily_wx[daily_wx["mean_wind_speed_ms"].fillna(99) < CALM_WIND_THRESHOLD_MS].copy()
calm_days["calm_flag"] = True

if "mean_no2" not in calm_days.columns:
    calm_days["mean_no2"] = np.nan
if "median_no2" not in calm_days.columns:
    calm_days["median_no2"] = np.nan

if not stats.empty and "mean_no2" in stats.columns:
    stats_daily = (
        stats.groupby(["site_id", "site_name", "date"], dropna=False)
        .agg(
            mean_no2=("mean_no2", "mean"),
            median_no2=("median_no2", "mean"),
        )
        .reset_index()
    )

    calm_days = calm_days.merge(
        stats_daily,
        on=["site_id", "site_name", "date"],
        how="left",
        suffixes=("", "_stats"),
    )

    if "mean_no2_stats" in calm_days.columns:
        calm_days["mean_no2"] = calm_days["mean_no2_stats"]
        calm_days = calm_days.drop(columns=["mean_no2_stats"])

    if "median_no2_stats" in calm_days.columns:
        calm_days["median_no2"] = calm_days["median_no2_stats"]
        calm_days = calm_days.drop(columns=["median_no2_stats"])

if "mean_no2" in calm_days.columns:
    pivot = calm_days.pivot_table(index="date", columns="site_id", values="mean_no2", aggfunc="mean")
else:
    pivot = pd.DataFrame()

if not pivot.empty and {"EST", "LWS"}.issubset(set(pivot.columns)):
    pivot["LWS_minus_EST"] = pivot["LWS"] - pivot["EST"]
if not pivot.empty and {"NHV", "EST"}.issubset(set(pivot.columns)):
    pivot["NHV_minus_EST"] = pivot["NHV"] - pivot["EST"]

calm_csv = Path(OUTPUT_DIR) / "wind_calm_candidate_days.csv"
pivot_csv = Path(OUTPUT_DIR) / "wind_calm_gradients.csv"

calm_days.to_csv(calm_csv, index=False)
pivot.reset_index().to_csv(pivot_csv, index=False)

print("Saved:", calm_csv)
print("Saved:", pivot_csv)

display(calm_days.head(20))
display(pivot.reset_index().head(20))


In [ ]:

fig, ax = plt.subplots(figsize=(10, 5))

plotted = False
if not pivot.empty:
    for col in ["LWS_minus_EST", "NHV_minus_EST"]:
        if col in pivot.columns and pivot[col].notna().any():
            ax.plot(pd.to_datetime(pivot.index), pivot[col], marker="o", label=col)
            plotted = True

ax.axhline(0, linestyle="--")
ax.set_title("Wind-calm inland-minus-coastal NO2 difference")
ax.set_xlabel("Date")
ax.set_ylabel("Difference")

if plotted:
    ax.legend()

fig.autofmt_xdate()
fig.tight_layout()

plot_path = Path(OUTPUT_DIR) / "wind_calm_difference_plot.png"
fig.savefig(plot_path, dpi=150)
plt.show()

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "experiment": "wind_calm_pollution_accumulation",
    "sites": SITES,
    "date_from": DATE_FROM,
    "date_to": DATE_TO,
    "calm_wind_threshold_ms": CALM_WIND_THRESHOLD_MS,
    "rows_weather": int(len(wx)),
    "rows_stats": int(len(stats)),
    "rows_calm_days": int(len(calm_days)),
    "rows_pivot": int(len(pivot)),
    "outputs": {
        "calm_days_csv": str(calm_csv),
        "gradient_csv": str(pivot_csv),
        "plot_png": str(plot_path),
    },
    "notes": [
        "If no scene-level NO2 stats are available, this notebook still produces calm-day screening outputs.",
        "Gradient columns may remain empty until 07c-style NO2 stats exist."
    ],
}
manifest_path = Path(OUTPUT_DIR) / "wind_calm_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Saved:", plot_path)
print("Saved:", manifest_path)
